# 03 · 自动检查：有毛病的代码存不进去

## 这一章在整个项目里的位置

```
00 总览 → 01 存档旧项目 ✅ → 02 新项目骨架 ✅ → [03 自动检查 ✅] → 04 GitHub 与 CI ✅ → 05 准备考题 ✅ → 06 评测集 ✅ → 07 判分 ✅ → 08 MLflow 评测 ✅ → 09 第一个基线 ✅ → 10 第一个工具 ✅ → 11 第二个工具 ✅ → 12 第三个工具 ✅ → 13 标准接口 MCP ✅ → 14 起：固定流程、agent …
```

> **这一章 10 月 7 日已经做完了。** 下面的命令会重新跑一遍检查、做三个小演示（演示文件用完马上删），不改项目里的任何东西。

## 背景：规矩靠人记，一定会忘

公司里一个项目很多人改，每天都有新代码进来。靠人盯着"有没有低级错误、有没有把密钥提交上去"是不可能的，所以代码库都有一套**自动把关**：

```
你改代码 → 提交前自动检查 → 推到 GitHub → 自动跑全部检查 → 合并
             (pre-commit，本章)              (CI，04 号)
```

旧项目也有自动检查，但只管一角：只检查 otovce_v2 一个目录，类型检查只检查其中一个子目录。原因是**检查是后补的**：补的时候，旧代码已经过不了检查，只好把范围缩小。

新项目**第一天就装好，而且开最严格的模式**：那时代码只有两行，没有历史包袱。以后每一行新代码都要过检查。

## 这一章做的 4 件事

| # | 做什么 | 用的技术 | 解决什么问题 |
|---|---|---|---|
| 1 | 查格式和低级错误 | ruff | 风格不统一、导入了不用的包、名字写错 |
| 2 | 查类型 | mypy | "数字加文字"这类错，要跑到那一行才发现 |
| 3 | 写测试 | pytest | 改了以后，不知道原来的功能坏没坏 |
| 4 | 提交前自动跑 | pre-commit | 靠人记会忘；密钥、大文件被提交 |

**花费：0。** 这些工具都在 02 号说的"只在开发时需要的"那一组里。

---

## 准备

运行下面这一格。它定义一个叫 `run` 的小工具，用来执行命令并显示结果。**不用看代码。**

> 运行 `uv` 的命令时，有时会先出现几行 `Building` / `Installed`：那是 uv 发现代码有变化，顺手更新了一下环境，可以忽略。

In [ ]:
import os, subprocess
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(ROOT)
os.environ.update(NO_COLOR="1", GH_PAGER="cat", GIT_PAGER="cat")
def run(cmd, check=True):
    print(f"$ {cmd}")
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(r.stdout + r.stderr)
    if check and r.returncode != 0:
        raise RuntimeError(f"failed: {cmd}")
print("项目目录:", ROOT)

---

## 第 1 步 · 查格式和低级错误：ruff

### 在干什么
用一个工具自动扫描所有代码：找出明显的错误，并把格式统一。

### 背景
常见的低级问题：导入了没用的包、变量名拼错（要等程序跑到那一行才报错）、每个人的缩进和换行习惯不一样。同事审查你的代码时，时间应该花在"逻辑对不对"上，而不是"这里多了个空格"。

### 用了什么技术，为什么
**ruff** 一个工具做两件事：
- `ruff check`：找问题（没用的导入、没定义的名字、容易出 bug 的写法）
- `ruff format`：自动统一格式

**为什么是 ruff**：它取代了以前要分开装的 flake8、black、isort 三个工具，快几十倍，现在是 Python 项目的主流选择。规则写在 `pyproject.toml` 的 `[tool.ruff]` 里，所有人共用一套。

### 公司里什么情况怎么做
- **检查报错**：大部分能用 `ruff check --fix` 自动修；修不了的，按提示改
- **团队觉得某条规则不合理**：改 `pyproject.toml` 里的规则，所有人一起生效，而不是各自忽略

### 执行
先检查整个项目，再故意写一段有问题的代码，看它能不能抓到（演示文件用完就删）：

In [ ]:
run("uv run ruff check && uv run ruff format --check")
Path("_demo.py").write_text("import os\nimport sys\n\n\ndef f(x):\n    return undefined_name + x\n")
run("uv run ruff check _demo.py --output-format concise", check=False)
Path("_demo.py").unlink()

### 结果怎么看
- 整个项目：`All checks passed!` 和 `... files already formatted`
- 演示代码：`Found 3 errors.`：`os`、`sys` 导入了没用（`F401`），`undefined_name` 没有定义（`F821`）。最后这个如果不检查，要等程序真正运行到那一行才会报错

### 面试怎么说
> "Ruff runs on every commit and in CI, so style issues and obvious bugs never reach code review."

---

## 第 2 步 · 查类型：mypy

### 在干什么
不运行程序，只读代码，检查"每个值的类型对不对得上"。

### 背景
Python 不强制写类型，很多错要等程序真正跑到那里才暴露。一个我们项目里的真实例子：05 号的病例里，有 69 例**没有致病基因**（比如染色体缺失的病例），`gene` 这个字段是空的。如果某处代码直接拿 `gene` 去做文字处理，程序要跑到那 69 例之一才会崩，可能是半小时以后。

### 用了什么技术，为什么
我们在代码里写上类型（比如"这个函数收两个整数，返回一个整数"），**mypy** 读代码时就核对每一处用法。`gene` 被标成"文字，或者空"，任何地方没先判断是不是空就使用它，mypy 在写代码时就报出来。

开的是**最严格的模式**（`strict = true`）：每个函数都必须写清类型。

### 公司里什么情况怎么做
- **新项目第一天开严格模式**；老项目一个目录一个目录地逐步加（旧项目就是这种情况，只加到了一个子目录）
- **mypy 报错**：多数是真问题，按提示改；确实是工具误判的，在那一行写明原因再忽略，而不是整个关掉

### 执行
故意写一个函数：说好返回整数，实际却把数字和文字相加：

In [ ]:
Path("_demo.py").write_text('def add(a: int, b: int) -> int:\n    return a + "1"\n')
run("uv run mypy _demo.py", check=False)
Path("_demo.py").unlink()

### 结果怎么看
`error: Unsupported operand types for + ("int" and "str")`：整数和文字不能相加。程序一次都没运行，错误就找到了。

### 面试怎么说
> "The codebase is type-checked with mypy in strict mode from day one, so whole classes of runtime errors, like using an optional field without checking it, are caught before the code runs."

---

## 第 3 步 · 写测试：改了以后，原来的功能还对不对

### 在干什么
写小程序，自动检查"代码做出来的结果是不是对的"。

### 背景
项目会一直改：加工具、加 agent、换模型。每改一次，都可能不小心弄坏原来能用的东西。每次都靠人工去试，迟早会漏。

### 用了什么技术，为什么
**pytest**：Python 的标准测试工具。测试放在 [`tests/`](../../tests/) 里，一条命令全部运行。每条测试就是"输入这个，应该得到那个"。

测试会随着项目一直增加（写这一章时 8 条，到第 07 章是 43 条）。按检查的东西分三类：

| 测试 | 检查什么 |
|---|---|
| 入口命令 | `raredx --help` 能用；缺了前一步时，提示先运行哪条命令；留出集不加 `--final` 不让跑 |
| 数据 | "查过但没有"的症状要去掉；没有基因的病例要保留；同一篇论文的病人永远分在同一边；被改过的文件要被认出来…… |
| 评测 | 清单冻结、留出集的病例混进开发集会被抓到（06 号）；判分规则、对照系统（07 号）；MLflow 评测和完整性检查（08 号） |
| 大模型基线 | 题目里没有答案、格式不对补救一次、思考模式的开关、密钥从哪里读（09 号）。用假的模型接口测，**不联网、不花钱** |

**测试不用真数据、不联网**：数据测试用的是在内存里现造的 3 例小病例。真数据 19 MB、要下载，GitHub 的检查机器上没有；现造的小数据哪里都能跑，而且可以专门造出"全是查过但没有的症状"、"没有基因"这些边角情况。

**以后会加更重要的一类：评测的回归测试。** 比如"agent 在固定的 100 例上的准确率，不能低于上一版"。改了 agent、分数掉了，测试就失败。

### 公司里什么情况怎么做
- **写一个新功能，同时写它的测试**
- **测试失败**：先弄清楚是"改坏了"，还是"本来就该变"（比如有意改了规则）。后者要在提交说明里写清原因，再改测试

### 执行

In [ ]:
run("uv run pytest -v")

### 结果怎么看
每个测试文件一行，一个点代表一条测试通过；最后一行 `… passed` 是当前的测试条数，全部通过，一两秒。

### 面试怎么说
> "Every behaviour has a unit test on small synthetic fixtures, so the suite runs anywhere in under a second; evaluation regressions will be gated the same way."

---

## 第 4 步 · 提交前自动跑：pre-commit

### 在干什么
设置一组检查，**每次 `git commit` 时自动运行**，有一项不通过，这次提交就不会发生。

### 背景
前三步的工具，如果要靠人每次记得手动跑，一定会忘。而且有些东西一旦提交就很麻烦：密钥被推到 GitHub、大数据文件进了记录、notebook 带着运行结果（结果里可能有电脑上的路径、邮箱、数据）。

### 用了什么技术，为什么
**pre-commit**：在 `git commit` 那一刻自动运行一组检查。配置写在 [`.pre-commit-config.yaml`](../../.pre-commit-config.yaml)：

| 检查 | 拦住什么 | 什么时候跑 |
|---|---|---|
| ruff | 低级错误、格式（第 1 步） | 每次提交 |
| mypy | 类型错误（第 2 步） | 每次提交 |
| 大文件 | 超过 500 KB 的文件（数据永远不该进记录） | 每次提交 |
| 私钥 | 密钥、证书 | 每次提交 |
| nbstripout | 自动清掉 notebook 的运行结果 | 每次提交 |
| 行尾空格、文件末尾等 | 没意义的格式差异 | 每次提交 |
| pytest | 测试（第 3 步） | **推送到 GitHub 之前** |

**为什么测试放在推送前，不是每次提交**：提交要快（几秒），人才愿意经常提交；测试会随着项目变慢。推送是"交作业"，那时跑刚好。

> **一个真实踩过的坑**：第一次演示时，mypy 这一项自己报了错：`Missing target module, package, files`。原因是它要检查哪些文件，原来写在另一个文件（`pyproject.toml`）里，而提交那一刻那个文件恰好不在场。改成在命令里直接写明 `uv run mypy src tests`，就不依赖别的文件了。

### 公司里什么情况怎么做
- **提交被拦住，提示 `files were modified by this hook`**：工具已经自动帮你修好了格式。重新 `git add`，再提交一次
- **提交被拦住，提示发现了密钥**：马上找到是哪个文件，把密钥挪到 `.env`。如果密钥已经推送出去过，**立刻去作废它**，删掉那次提交是不够的
- **真的急着提交**：`git commit --no-verify` 可以跳过检查，但 GitHub 上的自动检查（04 号）跳不过去

### 执行
先安装（重复安装是安全的）。然后演示：假装不小心把一把私钥放进了项目，看它会不会被拦住（演示文件马上删掉）：

In [ ]:
run("uv run pre-commit install --hook-type pre-commit --hook-type pre-push")
Path("leak_demo.txt").write_text("-----BEGIN " + "RSA PRIVATE KEY-----\nfake-key-for-demo\n")   # 拆成两半写，原因见下面
run("uv run pre-commit run --files leak_demo.txt", check=False)
Path("leak_demo.txt").unlink()

### 结果怎么看
- `pre-commit installed at .git/hooks/pre-commit`（还有 `pre-push`）：装好了
- 演示文件：`detect private key ... Failed`，并指出 `Private key found: leak_demo.txt`。真提交时，只要有一项 `Failed`，这次提交就不会发生
- 其他项显示 `Passed` 或 `Skipped`。`Skipped` 表示这一项不管这类文件，比如 mypy 只看 Python 文件

**为什么那句"私钥开头"要拆成两半写**：如果写成完整的一句，这个 notebook 文件本身就"含有私钥"了，提交时会被这个检查拦住。它只认内容、不管你是不是在演示，这正是它可靠的地方。

### 面试怎么说
> "Pre-commit hooks run ruff, mypy, secret detection and notebook output stripping on every commit, and the test suite before every push, so problems are caught before anything leaves the laptop."

---

## 本章小结

| 步骤 | 技术 | 公司里的作用 |
|---|---|---|
| 1 低级错误 | ruff | 风格统一，低级错误到不了审查 |
| 2 类型 | mypy（严格模式） | 运行之前就抓住一类错误 |
| 3 测试 | pytest | 改了以后，原来的功能没坏 |
| 4 自动跑 | pre-commit | 规矩靠机器，不靠人记；密钥和大文件出不去 |

**一句话**：从第一行代码起，每一次提交都要过检查，有毛病的存不进去。

前三步手动跑 ruff、mypy、pytest，是为了让你认识每个工具。**日常工作里不用手动跑**：提交时 pre-commit 自动跑，推送后 GitHub 自动跑。

## 面试题

1. **What's the difference between a linter and a type checker?**
   > "A linter like ruff catches style issues and suspicious patterns line by line; a type checker like mypy reasons about the types flowing through the program and catches mismatches, like adding a string to an int or using an optional value without checking it."

2. **Why run tests before push rather than on every commit?**
   > "Commits should take seconds so people commit often, and the test suite only grows. Running it at push time keeps commits fast, and CI runs it again regardless."

3. **A teammate accidentally committed an API key. What do you do?**
   > "Revoke and rotate the key first — once it's been pushed, assume it's compromised. Then remove it from the code, move it to environment configuration, and make sure a secret-detection hook is in place."

**下一章 04 · GitHub 与自动检查**：正本放到 GitHub，每个改动在一台干净的电脑上再检查一遍，绿了、你点合并才算数。